# How Do Avalanche Validators Actually Think About Staking Rewards?

| Field | Value |
| --- | --- |
| **Source** | [How Do Avalanche Validators Actually Think About Staking Rewards?](https://www.linkedin.com/pulse/how-do-avalanche-validators-actually-think-staking-rewards-lu-phd-q9igc) |
| **Author** | Siyu (Eric) Lu, PhD |
| **Published** | April 28, 2026 |
| **Status** | Work in progress |

This notebook reconstructs the article's evidence, data, models, and results while keeping every assumption and unresolved gap explicit.


In [1]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas
import pandas as pd
import panel as pn
import param as pm

from avalanche_research import datasets
from avalanche_research.catalog import dataset as catalog_dataset, load_catalog

pn.extension("tabulator")

In [2]:
class ArticleEvidence(pm.Parameterized):
    targets = pm.DataFrame(precedence=-1)

    @staticmethod
    def source():
        relative = Path("notebooks/validator-rewards-article-evidence.csv")
        for root in (Path.cwd().parent, Path.cwd()):
            candidate = root / relative
            if candidate.exists():
                return candidate
        raise FileNotFoundError(relative)

    def __init__(self, **params):
        params.setdefault("targets", pd.read_csv(self.source()).fillna(""))
        super().__init__(**params)

    def frame(self):
        return self.targets.copy()

    def view(self):
        return pn.widgets.Tabulator(
            self.frame(), show_index=False, disabled=True,
            frozen_columns=["target_id", "target"], pagination="local",
            page_size=len(self.targets), layout="fit_data_table",
            sizing_mode="stretch_width", height=860,
        )

article_evidence = ArticleEvidence(name="Article evidence")
article_evidence.view()


Tabulator(disabled=True, frozen_columns=['target_id', 'target'], height=860, page_size=40, pagination='local', show_index=False, sizing_mode='stretch_width', value=     target_id  ...)

## Evidence map

How complete is the evidence for each research domain at each stage of the reproduction?

In [3]:
ALL_EVIDENCE = "All"
EVIDENCE_STATES = (ALL_EVIDENCE, "Observed", "Reported", "Derived", "Provisional", "Unavailable")
EVIDENCE_STAGES = ("Source", "Data", "EDA", "Estimation", "Simulation", "Validation")
EVIDENCE_DOMAINS = ("Article", "Delegators", "Validators", "Market", "Protocol", "Outcomes")


def evidence_readiness(records):
    cells = pd.MultiIndex.from_product(
        [EVIDENCE_DOMAINS, EVIDENCE_STAGES], names=["domain", "stage"]
    )
    readiness = (
        records.assign(complete=records.status.eq("complete"))
        .groupby(["domain", "stage"])
        .agg(total=("target_id", "size"), complete=("complete", "sum"),
             target_names=("target", " | ".join))
        .reindex(cells, fill_value=0).reset_index()
    )
    readiness["target_names"] = readiness.target_names.replace(0, "")
    readiness["readiness"] = readiness.complete.div(readiness.total.where(readiness.total.gt(0)))
    readiness["label"] = [
        f"{complete}/{total}" if total else ""
        for complete, total in zip(readiness.complete, readiness.total)
    ]
    return readiness


class EvidenceMap(pm.Parameterized):
    evidence = pm.Selector(default=ALL_EVIDENCE, objects=EVIDENCE_STATES)
    targets = pm.DataFrame(precedence=-1)

    def records(self):
        selected = self.targets.evidence_state.eq(self.evidence.lower())
        return self.targets.copy() if self.evidence == ALL_EVIDENCE else self.targets[selected].copy()

    def frame(self):
        return evidence_readiness(self.records())

    @pm.depends("evidence")
    def figure(self):
        data = self.frame()
        heatmap = data.hvplot.heatmap(
            x="stage", y="domain", C="readiness", cmap="Blues", clim=(0, 1),
            clabel="Readiness", colorbar=True,
            title=f"{self.evidence} evidence readiness",
            hover_cols=["complete", "total", "target_names"],
        )
        return heatmap * hv.Labels(data, ["stage", "domain"], "label")

    @pm.depends("evidence")
    def table(self):
        return pn.widgets.Tabulator(
            self.records(), show_index=False, disabled=True,
            frozen_columns=["target_id", "target"], pagination="local", page_size=40,
            layout="fit_data_table", sizing_mode="stretch_width", height=330,
        )


evidence_map = EvidenceMap(targets=article_evidence.frame(), name="Evidence map")
pn.Column(evidence_map.param, evidence_map.figure, evidence_map.table)

Column
    [0] Column(margin=(5, 10), name='Evidence map')
        [0] StaticText(value='<b>Evidence map</b>')
        [1] Select(label='Evidence', name='Evidence', options=OrderedDict({'All': 'All',...]), value='All')
    [1] ParamMethod(method, _pane=HoloViews, defer_load=False)
    [2] ParamMethod(method, _pane=Tabulator, defer_load=False)

## Article data

What datasets are available, what does each row represent, and how can they be joined to reproduce the article?

In [4]:
ARTICLE_DATASETS = pd.DataFrame([
    ("Validator staking", "snowflake.primary-validator-stakes", "Accepted validator transaction", "accepted_at", "transaction_hash",
     "node_id; start_time; end_time; duration_days; stake_avax; delegation_fee_rate", "delegator.validator_transaction_hash; rewards.staking_transaction_hash; supply.transaction_hash", "VR-DATA-01"),
    ("Delegator staking", "snowflake.primary-delegator-stakes", "Accepted delegation transaction", "accepted_at", "transaction_hash",
     "node_id; validator_transaction_hash; delegator_address; start_time; end_time; duration_days; stake_avax", "validator.transaction_hash; rewards.staking_transaction_hash; supply.transaction_hash", "VR-DATA-02"),
    ("Reward outcomes", "snowflake.primary-staking-rewards", "Closed staking period", "rewarded_at", "reward_transaction_hash",
     "staking_transaction_hash; node_id; rewarded; potential_reward_avax; realized_reward_avax; fee_transfer_avax", "staking_transaction_hash to validator or delegator transaction_hash", "VR-DATA-03"),
    ("Supply context", "snowflake.primary-staking-supply", "Staking transaction with block supply", "accepted_at", "transaction_hash",
     "block_height; tx_block_position; previous_block_supply_navax; block_supply_navax; supply_before_navax", "transaction_hash to validator or delegator transaction_hash", "VR-DATA-04"),
    ("Price and volatility", "defillama.avax-price-daily", "Daily AVAX price observation", "observed_at", "observed_at",
     "observation_date; price_usd; log_return; realized_volatility_30d", "observation_date to staking calendar date", "VR-DATA-05"),
    ("Price and volatility", "coingecko.avax-market-daily", "Daily AVAX market observation", "observed_at", "observed_at",
     "observation_date; price_usd; market_cap_usd; volume_usd; log_return; realized_volatility_30d", "observation_date to staking calendar date; cross-check DefiLlama", "VR-DATA-05"),
], columns=["domain", "dataset_id", "observation_unit", "time_field", "primary_id", "key_fields", "joins", "target_id"])

In [5]:
def inspect_article_data(specs):
    catalog = load_catalog()
    connection = datasets.connect()
    rows = []
    try:
        for spec in specs.itertuples(index=False):
            definition = catalog_dataset(catalog, spec.dataset_id)
            available = datasets.runs(spec.dataset_id)
            if not available:
                # The data release leaves out the three largest Snowflake inputs. Their rows are inside derived.staking-periods.
                rows.append({"domain": spec.domain, "dataset_id": spec.dataset_id, "source": definition["source_id"],
                             "observation_unit": spec.observation_unit, "validation": "not in the data release",
                             "target_id": spec.target_id})
                continue
            run = available[0]
            view = datasets.view_name(spec.dataset_id)
            start, end, count = connection.execute(
                f'SELECT min("{spec.time_field}"), max("{spec.time_field}"), count(*) FROM "{view}"'
            ).fetchone()
            fields = {row[0] for row in connection.execute(f'DESCRIBE "{view}"').fetchall()}
            required = {spec.primary_id, *spec.key_fields.split("; ")}
            validation = "verified" if count == run["rows"] and required <= fields else "attention"
            rows.append({
                "domain": spec.domain,
                "dataset_id": spec.dataset_id,
                "source": definition["source_id"],
                "observation_unit": spec.observation_unit,
                "coverage": f"{pd.Timestamp(start).date()} to {pd.Timestamp(end).date()}",
                "rows": count,
                "primary_id": spec.primary_id,
                "key_fields": spec.key_fields,
                "joins": spec.joins,
                "catalog_status": definition["status"],
                "validation": validation,
                "target_id": spec.target_id,
            })
    finally:
        connection.close()
    return pd.DataFrame(rows)

In [6]:
class ArticleData(pm.Parameterized):
    contract = pm.DataFrame(precedence=-1)

    def __init__(self, **params):
        params.setdefault("contract", inspect_article_data(ARTICLE_DATASETS))
        super().__init__(**params)

    def frame(self):
        return self.contract.copy()

    def view(self):
        return pn.widgets.Tabulator(
            self.frame(), show_index=False, disabled=True,
            frozen_columns=["domain", "dataset_id"],
            layout="fit_data_table", sizing_mode="stretch_width",
        )


article_data = ArticleData(name="Article data")
article_data.view()

Tabulator(disabled=True, frozen_columns=['domain', 'dataset_id'], show_index=False, sizing_mode='stretch_width', value=              ...)

## Validator commitments

What constitutes one validator commitment, and which observed commitments form a valid analytical population?

In [7]:
VALIDATOR_DATASET = "snowflake.primary-validator-stakes"
DAYS_PER_WEEK = 7
DURATION_BOUNDS_DAYS = (14, 365)
DELEGATION_FEE_BOUNDS = (0.02, 1.00)
COMMITMENT_FIELDS = [
    "transaction_hash", "node_id", "accepted_at", "tx_type",
    "start_time", "end_time", "duration_days", "stake_avax",
    "delegation_fee_rate", "potential_reward_avax", "rewarded",
]
CORE_COMMITMENT_FIELDS = [
    "transaction_hash", "node_id", "accepted_at", "start_time",
    "end_time", "duration_days", "stake_avax", "delegation_fee_rate",
]


def prepare_validator_commitments():
    commitments = datasets.load(VALIDATOR_DATASET, run="validator-stakes-2026-09-19")[COMMITMENT_FIELDS].copy()
    commitments["duration_weeks"] = commitments.duration_days / DAYS_PER_WEEK
    commitments["structurally_valid"] = (
        commitments[CORE_COMMITMENT_FIELDS].notna().all(axis=1)
        & ~commitments.transaction_hash.duplicated(keep=False)
        & commitments.start_time.lt(commitments.end_time)
        & commitments.duration_days.between(*DURATION_BOUNDS_DAYS)
        & commitments.stake_avax.gt(0)
        & commitments.delegation_fee_rate.between(*DELEGATION_FEE_BOUNDS)
    )
    commitments["outcome_state"] = "structurally invalid"
    valid = commitments.structurally_valid
    commitments.loc[valid & commitments.rewarded.isna(), "outcome_state"] = "pending"
    commitments.loc[valid & commitments.rewarded.eq(True).fillna(False), "outcome_state"] = "rewarded"
    commitments.loc[valid & commitments.rewarded.eq(False).fillna(False), "outcome_state"] = "unrewarded"
    return commitments

In [8]:
class ValidatorCommitments(pm.Parameterized):
    commitments = pm.DataFrame(precedence=-1)

    def __init__(self, **params):
        params.setdefault("commitments", prepare_validator_commitments())
        super().__init__(**params)

    def frame(self):
        return self.commitments.copy()

    def audit(self):
        data = self.frame()
        valid = data.structurally_valid
        masks = [
            ("Source", "All accepted", pd.Series(True, index=data.index), "Every accepted validator staking transaction"),
            ("Exclusion", "Missing core field", data[CORE_COMMITMENT_FIELDS].isna().any(axis=1), "Required analytical field is null"),
            ("Exclusion", "Duplicate transaction", data.transaction_hash.duplicated(keep=False), "Transaction hash is not unique"),
            ("Exclusion", "Invalid chronology", data.start_time.ge(data.end_time), "Start is not earlier than end"),
            ("Exclusion", "Invalid duration", ~data.duration_days.between(*DURATION_BOUNDS_DAYS), "Duration is outside protocol bounds"),
            ("Exclusion", "Invalid stake", data.stake_avax.le(0), "Self-stake is not positive"),
            ("Exclusion", "Invalid fee", ~data.delegation_fee_rate.between(*DELEGATION_FEE_BOUNDS), "Fee is outside protocol bounds"),
            ("Partition", "Structurally invalid", ~valid, "Fails at least one structural check"),
            ("Partition", "Pending", data.outcome_state.eq("pending"), "Reward state is not yet observed"),
            ("Partition", "Rewarded", data.outcome_state.eq("rewarded"), "Completed with a positive reward decision"),
            ("Partition", "Unrewarded", data.outcome_state.eq("unrewarded"), "Completed without a reward"),
            ("Analysis", "Duration-choice population", valid, "Valid commitments, including pending commitments"),
            ("Analysis", "Reward-state population", valid & data.rewarded.notna(), "Valid commitments with observed reward state"),
            ("Data gap", "Missing potential reward", data.potential_reward_avax.isna(), "Potential reward is unavailable"),
        ]
        rows = []
        for kind, population, mask, rule in masks:
            selected = data.loc[mask]
            rows.append((kind, population, len(selected), len(selected) / len(data),
                         selected.node_id.nunique(), selected.accepted_at.min(), selected.accepted_at.max(), rule))
        return pd.DataFrame(rows, columns=[
            "kind", "population", "rows", "share", "nodes", "accepted_from", "accepted_to", "rule",
        ])

    def view(self):
        return pn.widgets.Tabulator(
            self.audit(), show_index=False, disabled=True,
            frozen_columns=["kind", "population"],
            layout="fit_data_table", sizing_mode="stretch_width",
        )


validator_commitments = ValidatorCommitments(name="Validator commitments")
validator_commitments.view()

Tabulator(disabled=True, frozen_columns=['kind', 'population'], show_index=False, sizing_mode='stretch_width', value=         kind             ...)

## Observed commitment durations

How are structurally valid validator commitments distributed across duration? Each accepted commitment receives equal weight.

In [9]:
DURATION_BIN_WEEKS = 1


class ValidationDurations(pm.Parameterized):
    measure = pm.Selector(default="Percent", objects=["Percent", "Value"])
    commitments = pm.DataFrame(precedence=-1)

    def population(self):
        return self.commitments.loc[self.commitments.structurally_valid, "duration_weeks"]

    def frame(self):
        durations = self.population()
        lower = DURATION_BOUNDS_DAYS[0] // DAYS_PER_WEEK
        upper = (DURATION_BOUNDS_DAYS[1] + DAYS_PER_WEEK - 1) // DAYS_PER_WEEK
        intervals = pd.cut(durations, bins=range(lower, upper + 1), right=False)
        frame = intervals.value_counts(sort=False).rename_axis("duration_interval").reset_index(name="commitments")
        frame["duration_week"] = frame.duration_interval.map(lambda interval: interval.mid).astype(float)
        frame["share_percent"] = 100 * frame.commitments / len(durations)
        frame["duration_interval"] = frame.duration_interval.astype(str)
        return frame[["duration_week", "duration_interval", "commitments", "share_percent"]]

    def statistics(self):
        durations = self.population()
        return {"mean": durations.mean(), "median": durations.median()}

    def reference(self, name, duration, maximum, y):
        data = [(duration, 0, name, duration), (duration, maximum, name, duration)]
        return hv.Curve(data, kdims=["duration_week"], vdims=[y, "statistic", "value_weeks"],
                        label=f"{name} · {duration:.2f} weeks")

    @pm.depends("measure")
    def view(self):
        frame, stats = self.frame(), self.statistics()
        y = "share_percent" if self.measure == "Percent" else "commitments"
        ylabel = "Share of commitments (%)" if self.measure == "Percent" else "Number of commitments"
        maximum = frame[y].max()
        x_maximum = (DURATION_BOUNDS_DAYS[1] + DAYS_PER_WEEK - 1) // DAYS_PER_WEEK
        bars = frame.hvplot.bar(
            x="duration_week", y=y,
            hover_cols=["duration_interval", "commitments", "share_percent"],
            xlabel="Validation duration (weeks)", ylabel=ylabel,
            title="Observed validation commitment durations",
            color="#3B82B4", alpha=0.85, height=420, responsive=True,
        )
        mean = self.reference("Mean", stats["mean"], maximum, y)
        median = self.reference("Median", stats["median"], maximum, y)
        tooltips = [("Statistic", "@statistic"), ("Duration", "@value_weeks{0.00} weeks")]
        return (bars * mean.opts(color="#D95F02", line_dash="dashed", line_width=2, tools=["hover"], hover_tooltips=tooltips)
                * median.opts(color="#1B9E77", line_dash="dotted", line_width=2, tools=["hover"], hover_tooltips=tooltips)).opts(
            legend_position="top_right", padding=0,
            xlim=(0, x_maximum), ylim=(0, maximum),
        )


validation_durations = ValidationDurations(commitments=validator_commitments.frame(), name="Validation durations")
pn.Column("One-week bins show the empirical distribution before smoothing or stake weighting.",
          pn.Param(validation_durations, parameters=["measure"], show_name=False), validation_durations.view)


Column
    [0] Markdown(str)
    [1] Param(ValidationDurations, name='Validation durations', parameters=['measure'], show_name=False)
    [2] ParamMethod(method, _pane=HoloViews, defer_load=False)

## Stake by commitment duration

Where is validator self-stake concentrated across commitment lengths? This sums self-stake across accepted commitments and therefore measures cumulative commitments, not concurrent network stake.

In [10]:
AVAX_PER_MILLION = 1_000_000


class StakeByDuration(pm.Parameterized):
    measure = pm.Selector(default="Percent", objects=["Percent", "Value"])
    commitments = pm.DataFrame(precedence=-1)

    def population(self):
        return self.commitments.loc[self.commitments.structurally_valid, ["duration_weeks", "stake_avax"]].copy()

    def frame(self):
        data = self.population()
        lower = DURATION_BOUNDS_DAYS[0] // DAYS_PER_WEEK
        upper = (DURATION_BOUNDS_DAYS[1] + DAYS_PER_WEEK - 1) // DAYS_PER_WEEK
        intervals = pd.cut(data.duration_weeks, bins=range(lower, upper + 1), right=False)
        frame = (data.assign(duration_interval=intervals)
                 .groupby("duration_interval", observed=False)
                 .agg(commitments=("stake_avax", "size"), stake_avax=("stake_avax", "sum")).reset_index())
        frame["duration_week"] = frame.duration_interval.map(lambda interval: interval.mid).astype(float)
        frame["stake_millions"] = frame.stake_avax / AVAX_PER_MILLION
        frame["stake_share_percent"] = 100 * frame.stake_avax / frame.stake_avax.sum()
        frame["duration_interval"] = frame.duration_interval.astype(str)
        return frame[["duration_week", "duration_interval", "commitments", "stake_avax", "stake_millions", "stake_share_percent"]]

    def statistics(self):
        data = self.population().sort_values("duration_weeks")
        total = data.stake_avax.sum()
        mean = (data.duration_weeks * data.stake_avax).sum() / total
        median = data.loc[data.stake_avax.cumsum().ge(total / 2), "duration_weeks"].iloc[0]
        return {"mean": mean, "median": median}

    def reference(self, name, duration, maximum, y):
        data = [(duration, 0, name, duration), (duration, maximum, name, duration)]
        return hv.Curve(data, kdims=["duration_week"], vdims=[y, "statistic", "value_weeks"],
                        label=f"Stake-weighted {name.lower()} · {duration:.2f} weeks")

    @pm.depends("measure")
    def view(self):
        frame, stats = self.frame(), self.statistics()
        y = "stake_share_percent" if self.measure == "Percent" else "stake_millions"
        ylabel = "Share of cumulative self-stake (%)" if self.measure == "Percent" else "Cumulative self-stake committed (million AVAX)"
        maximum = frame[y].max()
        x_maximum = (DURATION_BOUNDS_DAYS[1] + DAYS_PER_WEEK - 1) // DAYS_PER_WEEK
        bars = frame.hvplot.bar(x="duration_week", y=y, hover_cols=["duration_interval", "commitments", "stake_avax", "stake_millions", "stake_share_percent"],
                                xlabel="Validation duration (weeks)", ylabel=ylabel,
                                title="Validator self-stake by commitment duration", color="#3B82B4", alpha=0.85, height=420, responsive=True)
        tooltips = [("Statistic", "@statistic"), ("Duration", "@value_weeks{0.00} weeks")]
        mean = self.reference("Mean", stats["mean"], maximum, y).opts(color="#D95F02", line_dash="dashed", line_width=2, tools=["hover"], hover_tooltips=tooltips)
        median = self.reference("Median", stats["median"], maximum, y).opts(color="#1B9E77", line_dash="dotted", line_width=2, tools=["hover"], hover_tooltips=tooltips)
        return (bars * mean * median).opts(legend_position="top_right", padding=0, xlim=(0, x_maximum), ylim=(0, maximum))


stake_by_duration = StakeByDuration(commitments=validator_commitments.frame(), name="Stake by duration")
pn.Column("Each bar sums validator self-stake across accepted commitments in that duration bin.",
          pn.Param(stake_by_duration, parameters=["measure"], show_name=False), stake_by_duration.view)


Column
    [0] Markdown(str)
    [1] Param(StakeByDuration, name='Stake by duration', parameters=['measure'], show_name=False)
    [2] ParamMethod(method, _pane=HoloViews, defer_load=False)

## Upgrade periods

Which upgrade period contained each validator's commitment decision? Assign periods at `accepted_at` in UTC using the checked `derived.staking-periods` regime. All Apricot protocol phases are combined into one Apricot upgrade period; the exact phase remains available in the underlying `regime` field. Periods are half-open: the activation instant belongs to the new period. The table counts every accepted commitment and reports duration statistics only for structurally valid commitments. This is an observation grouping, not evidence that an upgrade caused a change. `duration_weeks` is the interval between scheduled boundaries; Genesis has no recorded start boundary, so its duration is blank. `total_stake_avax` sums self-stake across valid accepted commitments, not concurrent network stake. Stake means and medians use one observation per valid commitment; weighted duration statistics use each commitment's self-stake as its weight.

In [11]:
from avalanche_research.model.protocol import MAINNET_UPGRADES

APRICOT_PERIOD = "apricot"

class UpgradePeriods(pm.Parameterized):
    commitments = pm.DataFrame(precedence=-1)
    def assigned(self):
        with datasets.connect() as connection:
            lookup = connection.execute(
                "SELECT transaction_hash, regime FROM staking_periods WHERE role = 'validator'"
            ).fetchdf()
        rows = self.commitments.merge(lookup, on="transaction_hash", how="left", validate="one_to_one")
        if rows.regime.isna().any():
            raise ValueError("Some validator commitments lack a checked upgrade-period assignment")
        rows["period"] = rows.regime.mask(
            rows.regime.str.startswith(APRICOT_PERIOD), APRICOT_PERIOD
        )
        return rows

    def frame(self):
        rows = self.assigned()
        valid = rows.loc[rows.structurally_valid].copy()
        valid["stake_weeks"] = valid.stake_avax * valid.duration_weeks
        counts = rows.groupby("period").transaction_hash.size().rename("commitments")
        stats = valid.groupby("period").agg(
            valid_commitments=("transaction_hash", "size"),
            mean_weeks=("duration_weeks", "mean"),
            median_weeks=("duration_weeks", "median"),
            total_stake_avax=("stake_avax", "sum"),
            mean_stake_avax=("stake_avax", "mean"),
            median_stake_avax=("stake_avax", "median"),
            stake_weeks=("stake_weeks", "sum"),
        )
        stats["stake_weighted_mean_weeks"] = stats.stake_weeks / stats.total_stake_avax
        ordered = valid.sort_values(["period", "duration_weeks"])
        running_stake = ordered.groupby("period").stake_avax.cumsum()
        half_stake = ordered.groupby("period").stake_avax.transform("sum") / 2
        stats["stake_weighted_median_weeks"] = ordered.loc[running_stake.ge(half_stake)].groupby("period").duration_weeks.first()
        summary = counts.to_frame().join(stats.drop(columns="stake_weeks")).rename_axis("period").reset_index()
        starts = [("genesis", pd.NaT)]
        starts += [(u.name, u.activated_at.replace(tzinfo=None)) for u in MAINNET_UPGRADES.upgrades]
        schedule = pd.DataFrame(starts, columns=["period", "starts_utc"])
        schedule["ends_utc"] = schedule.starts_utc.shift(-1)
        schedule["period"] = schedule.period.mask(
            schedule.period.str.startswith(APRICOT_PERIOD), APRICOT_PERIOD
        )
        schedule = schedule.groupby("period", sort=False).agg(
            starts_utc=("starts_utc", "min"), ends_utc=("ends_utc", "max")
        ).reset_index()
        schedule["duration_weeks"] = (schedule.ends_utc - schedule.starts_utc) / pd.to_timedelta(1, unit="W")
        return schedule.merge(summary, on="period", how="inner").round(2)

    def view(self):
        return pn.widgets.Tabulator(
            self.frame(), show_index=False, disabled=True, frozen_columns=["period"],
            layout="fit_data_table", sizing_mode="stretch_width",
        )


upgrade_periods = UpgradePeriods(commitments=validator_commitments.frame(), name="Upgrade periods")
upgrade_periods.view()

Tabulator(disabled=True, frozen_columns=['period'], show_index=False, sizing_mode='stretch_width', value=    period    ...)

## Observed duration by upgrade period

Within each upgrade period, what share of accepted validator commitments chose each commitment duration? All Apricot protocol phases are combined into one row, while the exact phase remains in the underlying data. Each row is normalized independently. Period labels identify observation windows, not causal effects; the first and final periods have incomplete data boundaries.

In [12]:
DURATION_WEEK_LIMIT = 53
DURATION_BREAKS = range(DURATION_WEEK_LIMIT + 1)

class UpgradeDurationMap(pm.Parameterized):
    commitments = pm.DataFrame(precedence=-1)
    periods = pm.DataFrame(precedence=-1)

    def frame(self):
        data = self.commitments.loc[
            self.commitments.structurally_valid, ["period", "duration_weeks"]
            ].copy()
        data["duration_bin"] = pd.cut(data.duration_weeks, DURATION_BREAKS, right=False)
        order = self.periods.period.tolist()
        cells = pd.MultiIndex.from_product(
            [order, data.duration_bin.cat.categories], names=["period", "duration_bin"]
        )
        frame = (
            data.groupby(["period", "duration_bin"], observed=False).size()
            .reindex(cells, fill_value=0).rename("commitments").reset_index()
        )
        frame["period_total"] = frame.groupby("period").commitments.transform("sum")
        frame["share_percent"] = 100 * frame.commitments / frame.period_total
        frame["duration_week"] = frame.duration_bin.map(lambda value: value.mid).astype(float)
        frame["duration_interval"] = frame.duration_bin.astype(str)
        labels = self.periods.period.str.replace("_", " ").str.title().tolist()
        frame["period_label"] = pd.Categorical(
            frame.period.str.replace("_", " ").str.title(), labels, ordered=True
        )
        return frame.merge(self.periods, on="period", how="left")

    def view(self):
        figure = self.frame().hvplot.heatmap(
            x="duration_week", y="period_label", C="share_percent", cmap="Blues",
            clabel="Share of period commitments (%)", colorbar=True,
            hover_cols=["period", "duration_interval", "commitments", "period_total", "starts_utc", "ends_utc"],
            xlabel="Validation duration (weeks)", ylabel="",
            title="Observed commitment durations by upgrade period",
            height=520, responsive=True,
        ).opts(invert_yaxis=True, xlim=(0, DURATION_WEEK_LIMIT), padding=0)
        return pn.Column(figure, sizing_mode="stretch_width")

duration_by_upgrade = UpgradeDurationMap(
    commitments=upgrade_periods.assigned(),
    periods=upgrade_periods.frame()[["period", "starts_utc", "ends_utc"]],
    name="Duration by upgrade period",
)
duration_by_upgrade.view()

Column(sizing_mode='stretch_width')
    [0] HoloViews(HeatMap, height=520, sizing_mode='stretch_width')

In [13]:
import numbers

class ValidatorCommitmentComparison(pm.Parameterized):
    commitments = pm.DataFrame(precedence=-1)
    def measures(self, rows, start, end):
        count, nodes = len(rows), rows.node_id.nunique()
        stake, weeks = rows.stake_avax.sum(), rows.duration_weeks
        avax_weeks = (rows.stake_avax * weeks).sum()
        ordered = rows.sort_values("duration_weeks")
        weighted_median = ordered.loc[ordered.stake_avax.cumsum().ge(stake / 2), "duration_weeks"].min() if stake else float("nan")
        return pd.Series({
            "Protocol window start (UTC)": start,
            "Protocol window end (UTC)": end,
            "Protocol window length (weeks)": (end - start) / pd.to_timedelta(1, unit="W"),
            "Validator commitments": count,
            "Distinct node IDs": nodes,
            "Commitments per node ID": count / nodes if nodes else float("nan"),
            "Gross committed self-stake (AVAX)": stake,
            "Mean self-stake per commitment (AVAX)": stake / count if count else float("nan"),
            "Median self-stake per commitment (AVAX)": rows.stake_avax.median(),
            "Mean commitment duration (weeks)": weeks.mean(),
            "Median commitment duration (weeks)": weeks.median(),
            "Gross committed self-stake-duration (AVAX-weeks)": avax_weeks,
            "Mean self-stake-duration per commitment (AVAX-weeks)": avax_weeks / count if count else float("nan"),
            "Self-stake-weighted mean duration (weeks)": avax_weeks / stake if stake else float("nan"),
            "Self-stake-weighted median duration (weeks)": weighted_median,
        })

    def frame(self):
        rows = self.commitments.loc[self.commitments.structurally_valid]
        periods = upgrade_periods.frame().set_index("period")
        periods.loc["genesis", "starts_utc"] = rows.accepted_at.min()
        values = {"All data": self.measures(rows, rows.accepted_at.min(), periods.loc["granite", "ends_utc"])}
        for era, period in periods.iterrows():
            values[era.title()] = self.measures(rows.loc[rows.period.eq(era)], period.starts_utc, period.ends_utc)
        return pd.DataFrame(values).T.rename_axis("Era").reset_index()
    @staticmethod
    def format_value(value):
        if isinstance(value, pd.Timestamp): return value.strftime("%Y-%m-%d %H:%M UTC")
        if isinstance(value, numbers.Real): return f"{value:,.0f}" if isinstance(value, numbers.Integral) else f"{value:,.2f}"
        return value
    def view(self):
        table = pn.widgets.Tabulator(self.frame().map(self.format_value), show_index=False,
                                     disabled=True, frozen_columns=["Era"], layout="fit_data_table", sizing_mode="stretch_width")
        cutoff = self.commitments.loc[self.commitments.structurally_valid, "accepted_at"].max()
        return pn.Column("### Validator commitments by protocol era", table,
                         f"Data cutoff: {self.format_value(cutoff)}. Granite is partially observed. "
                         "Node IDs do not identify operators. Gross stake sums commitments, not simultaneous active stake.")
commitment_comparison = ValidatorCommitmentComparison(commitments=upgrade_periods.assigned())
commitment_comparison.view()

Column
    [0] Markdown(str)
    [1] Tabulator(disabled=True, frozen_columns=['Era'], show_index=False, sizing_mode='stretch_width', value=        Era Protocol windo...)
    [2] Markdown(str)

In [14]:
LEDGER_DATASET = "derived.avax-monetary-daily"
WEI_PER_AVAX = 10**18


class CommitmentsWithSupply(pm.Parameterized):
    """The era table above with realized total supply, active stake, and the two staking ratios (D1, D2, D5, D6 in the monetary ledger plan).

    Each day is observed at 23:59:59 UTC and belongs to the era holding that instant. Supply is the ledger's realized
    close, written only for checked days, and stake is checked over more days than supply, so each mean covers the
    days shown in its coverage column."""
    comparison = pm.ClassSelector(class_=ValidatorCommitmentComparison, precedence=-1)

    def daily(self):
        with datasets.connect() as connection:
            return connection.execute(f"""
                SELECT date_utc::TIMESTAMP + INTERVAL '23:59:59' AS observed_at,
                       total_supply_close_wei::DOUBLE / {WEI_PER_AVAX} AS supply_avax,
                       CASE WHEN stake_coverage = 'complete' THEN active_validator_stake_wei::DOUBLE / {WEI_PER_AVAX} END AS validator_stake_avax,
                       CASE WHEN stake_coverage = 'complete' THEN active_delegated_stake_wei::DOUBLE / {WEI_PER_AVAX} END AS delegated_stake_avax,
                       active_validator_stake_wei::DOUBLE / total_supply_close_wei::DOUBLE AS validator_ratio,
                       (active_validator_stake_wei + active_delegated_stake_wei)::DOUBLE / total_supply_close_wei::DOUBLE AS combined_ratio
                FROM avax_monetary_daily
            """).fetchdf()

    def frame(self):
        table, daily = self.comparison.frame(), self.daily()
        added = []
        for start, end in zip(table["Protocol window start (UTC)"], table["Protocol window end (UTC)"]):
            days = daily.loc[daily.observed_at.ge(start) & daily.observed_at.lt(end)]
            checked, staked = days.dropna(subset=["supply_avax"]), days.dropna(subset=["validator_stake_avax"])
            added.append({
                "Mean total supply (AVAX)": checked.supply_avax.mean(),
                "Mean validator stake (AVAX)": staked.validator_stake_avax.mean(),
                "Mean delegator stake (AVAX)": staked.delegated_stake_avax.mean(),
                "Mean validator staking ratio (%)": 100 * checked.validator_ratio.mean(),
                "Mean combined staking ratio (%)": 100 * checked.combined_ratio.mean(),
                "Days with checked supply": f"{len(checked):,} of {len(days):,}",
                "Days with checked stake": f"{len(staked):,} of {len(days):,}",
            })
        return pd.concat([table, pd.DataFrame(added)], axis=1)

    def view(self):
        run = datasets.receipt(LEDGER_DATASET)["run"]
        table = pn.widgets.Tabulator(self.frame().map(self.comparison.format_value), show_index=False,
                                     disabled=True, frozen_columns=["Era"], layout="fit_data_table", sizing_mode="stretch_width")
        return pn.Column("### Validator commitments by protocol era, with supply and staking ratio", table,
                         f"Supply and stake from `{LEDGER_DATASET}`, run `{run}`. Total supply is realized: genesis plus rewards paid "
                         "minus burns on the P-, X-, and C-Chains, never the P-Chain protocol counter. Validator stake is the self-stake of "
                         "active validators and delegator stake the stake delegated to them, in total, not per validator. The validator "
                         "ratio is validator stake over supply, and the combined ratio adds delegator stake. Each is the mean of daily "
                         "readings at 23:59:59 UTC, over the days its coverage column counts.")


commitments_with_supply = CommitmentsWithSupply(comparison=commitment_comparison)
commitments_with_supply.view()

Column
    [0] Markdown(str)
    [1] Tabulator(disabled=True, frozen_columns=['Era'], show_index=False, sizing_mode='stretch_width', value=        Era Protocol windo...)
    [2] Markdown(str)